# 🎯 [MNT10_M09] AAR 기반 출동 우선순위 배정 오류 분석 및 가중치 매트릭스 재튜닝 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MNT10/mnt10_m09_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장비정비·수리부속 (Level 3) |
| **미션 ID** | MNT10_M09 |
| **미션 주제** | AAR 기반 출동 우선순위 배정 오류 분석 및 가중치 매트릭스 재튜닝 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
야전에서 장비가 고장 나면 이동정비반이 얼마나 빨리 도착해 고치느냐가 부대 전투력 회복 속도를 좌우하지만, 가까운 정비반부터 보내는 방식은 요청이 몰리는 지역의 정비반에 부하를 집중시키고 핵심 장비를 뒤로 미루는 문제가 있다. 본 프로젝트는 정비인력의 부하율과 공정 균형을 데이터로 예측·관리하는 것을 목표로, 교육용 가상 지역의 긴급 정비 요청과 이동정비반 자료를 코드로 만들어 분석한다. 요청 데이터 스키마 정의, 정비반 상태 자료 정제, 거리 기준 Baseline과 탐색적 분석, 중요도·정비 소요시간·이동 경로를 종합한 우선순위 엔진, 출동 지연 위험도와 현장 수리 가능성 점수, 관제 대시보드와 설명가능성, 정비대대장 승인(HITL) 시뮬레이터, 동시 다발 요청 스트레스 시험, AAR 가중치 재튜닝과 효과 산출, 재학습을 포함한 통합 도구까지 단계별로 구현한다. 최종 목표는 핵심 장비의 가동중단 시간을 줄이면서 정비반 간 부하를 고르게 하고, 출동 순서 변경과 재배치는 지휘관이 근거를 보고 결정하도록 지원하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 📁 프로젝트 산출물(deliverables) 디렉토리 구성 안내

본 과제는 실무 현장에서 바로 쓰일 수 있는 국방 소프트웨어 아키텍처 표준을 지향합니다.  
과제를 진행하며 생성되는 산출물은 아래 규칙에 따라 각 폴더에 체계적으로 저장해 주세요.

| 디렉토리 | 군수 실무 비유 | 주 역할 및 산출물 파일 | 분리 목적 |
| :--- | :--- | :--- | :--- |
| **`docs/`** | 야전 교범 / 정비 지침서 | `mnt10_logistics_domain_spec.md` (미션1), `mnt10_operations_guide.md` (미션10) | 코드를 모르는 지휘관/후임자도 데이터 규격과 비상 매뉴얼을 이해하도록 보장 |
| **`notebooks/`** | 정비 시험실 / 연구실 | `mnt10_eda_analysis.ipynb` (미션3), 본 실습 노트북 | AI와 대화하며 자유롭게 데이터를 시각화하고 알고리즘을 반복 실험 |
| **`src/`** | 실제 장착 핵심 부품 모듈 | `src/etl/`, `src/features/`, `src/models/`, `src/dashboard/`, `src/optimization/` | 운영 서버에서 자동 배치 실행 가능한 순수 파이썬 모듈(`.py`)로 부품화 |
| **`reports/`** | 지휘관 결심 보고서 | `mnt10_roi_analysis.md` (미션9), AAR 오배정 분석서 | 정비대대장/군수참모에게 정비 지연 15% 감축 등 정량 ROI 성과 보고 |
| **`pipeline/`** | 자동화 컨베이어 벨트 | `pipeline/mnt10_retrain.py` (미션10) | 긴급 요청 유입 시 전처리부터 관제까지 자동 구동 및 재학습 파이프라인 |

> **💡 [바이브 코딩(AI 협업) 실습 요령]**  
> 1. **노트북에서 먼저 검증**: 본 노트북의 각 미션 셀에서 AI에게 질의하며 단계별 코드를 실행하고 결과를 확인합니다.  
> 2. **AI에게 모듈화 요청**: 검증 완료 후 AI에게 *"이 코드를 `src/.../` 파일에 넣을 수 있게 함수/클래스로 모듈화해줘"*라고 요청합니다.  
> 3. **디렉토리별 저장**: 제안된 코드를 해당 파일에 저장하여 자연스럽게 실무 패키지를 완성합니다.

---



In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 09. AAR 기반 출동 우선순위 배정 오류 분석 및 가중치 매트릭스 재튜닝
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

AAR 기반 출동 우선순위 배정 오류를 분석하고 가중치를 재튜닝한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **출동 우선순위 배정 결과를 AAR로 검토해 오류 원인을 분석하고, 가중치 매트릭스를 재튜닝해 도입 효과를 정량 지표로 산출한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **핵심장비 도착 4시간 초과 사례를 정비반 부하 집중·원거리 배정·도로·기상 지연·배정 주기 대기로 분류 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `reports/mnt10_roi_analysis.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 추정 산식의 객관성 및 군 재정/군수 기획 부서 설득력

#### 미션 PASS 조건

- 정량 기준: 비용 절감액, 시간 단축률 등 3대 정량 효과 지표 산출


- 진행 기준: ROI 분석 보고서 완성 및 주요 지표 민감도 분석 수록


- 검증 방법: 회계/기획 기준 모델과의 비교 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""출동 우선순위 배정 결과를 사후검토(AAR)해 핵심장비 지연 사례의 원인(정비반 부하 집중·원거리 배정·도로·기상 지연 등)을 분류하고, 과거 기간으로 우선순위·배정 가중치 매트릭스를 재튜닝해 평가 기간에 검증한다. 가동중단 비용·핵심장비 도착시간·정비인력 초과근무의 3대 정량 효과 지표와 ROI 요약을 출력한다."""
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor

# ---------------------------------------------------------------------------
# [교육용가상자료] 야전 이동정비반·긴급 정비 요청 생성기 (실제 부대·위치·장비·인원 정보 없음)
# 60km x 60km 가상 지역에 이동정비반 6개와 30일간의 긴급 정비 요청을 만든다.
# 장비 구분(mobility/fires/comms/power)과 고장 유형은 교육용 추상 분류다.
# ---------------------------------------------------------------------------
AREA_KM, DUTY_H = 60, 10.0                     # 가상 작전지역 크기, 정비반 1일 기준 가용시간
FAULT_H = {"engine": 6.0, "drivetrain": 5.0, "hydraulic": 3.0, "electric": 2.0, "comm_module": 1.5, "cooling": 2.5}
FIELD_P = {"engine": 0.3, "drivetrain": 0.35, "hydraulic": 0.75, "electric": 0.9, "comm_module": 0.95, "cooling": 0.8}
EQUIP_F = {"mobility": 1.0, "fires": 1.2, "comms": 0.7, "power": 0.8}
FAULT_OF = {"mobility": [0.3, 0.25, 0.2, 0.1, 0.0, 0.15], "fires": [0.2, 0.15, 0.35, 0.15, 0.0, 0.15],
            "comms": [0.0, 0.0, 0.0, 0.4, 0.5, 0.1], "power": [0.35, 0.0, 0.0, 0.35, 0.0, 0.3]}
CRIT_P = {"mobility": [0.45, 0.35, 0.2], "fires": [0.2, 0.35, 0.45], "comms": [0.3, 0.4, 0.3], "power": [0.6, 0.3, 0.1]}
HUBS = [(15, 20), (45, 25), (30, 50)]           # 가상 훈련장 3곳(요청이 몰리는 지점)


def make_teams(seed=0):
    rng = np.random.default_rng(seed)
    base = [(8, 10), (30, 6), (52, 12), (10, 44), (32, 38), (54, 50)]
    skills = [["fires", "mobility"], ["comms", "mobility"], ["fires", "power"],
              ["mobility", "power"], ["comms", "fires"], ["comms", "mobility", "power"]]
    return [{"team": f"MT{i + 1}", "x": float(x), "y": float(y), "crew": int(rng.integers(3, 6)), "skills": s}
            for i, ((x, y), s) in enumerate(zip(base, skills))]


def make_requests(n_days=30, per_day=10, seed=1, surge_days=(8, 9, 10, 11, 22, 23, 24)):
    """하루 평균 per_day건, 훈련 주간(surge_days)에는 1.6배의 긴급 정비 요청을 만든다."""
    rng = np.random.default_rng(seed)
    rows = []
    for d in range(n_days):
        n = rng.poisson(per_day * (1.6 if d in surge_days else 1.0))
        weather = rng.choice(["clear", "rain", "snow"], p=[0.65, 0.25, 0.1])
        for _ in range(n):
            if rng.random() < 0.7:
                hx, hy = HUBS[rng.integers(0, 3)]
                x, y = np.clip(rng.normal(hx, 6), 0, AREA_KM), np.clip(rng.normal(hy, 6), 0, AREA_KM)
            else:
                x, y = rng.uniform(0, AREA_KM, 2)
            eq = str(rng.choice(list(EQUIP_F), p=[0.45, 0.25, 0.2, 0.1]))
            fault = str(rng.choice(list(FAULT_H), p=FAULT_OF[eq]))
            parts = bool(rng.random() < 0.75)
            bad = weather != "clear"
            std_h = FAULT_H[fault] * EQUIP_F[eq] * (1.35 if not parts else 1.0) * (1.15 if bad else 1.0) * rng.lognormal(0, 0.12)
            rows.append({"t_req": d * 24 + float(np.clip(rng.normal(13, 4), 5, 23)), "day": d,
                         "x": float(x), "y": float(y), "equip": eq, "fault": fault,
                         "crit": int(rng.choice([1, 2, 3], p=CRIT_P[eq])), "parts_ready": parts,
                         "weather": str(weather), "road": "unpaved" if y > 30 and rng.random() < 0.7 else "paved",
                         "field_ok": bool(rng.random() < FIELD_P[fault] * (1.0 if parts else 0.5)),
                         "std_repair_h": round(float(std_h), 2), "road_delay": float(rng.lognormal(0, 0.15))})
    df = pd.DataFrame(rows).sort_values("t_req").reset_index(drop=True)
    df.insert(0, "req_id", [f"R{i:04d}" for i in range(len(df))])
    return df
# ---------------------------------------------------------------------------

# ---- 출동 배정·정비 모의 공통 함수 ----
W0 = {"crit": 1.0, "wait": 0.3, "short": 0.2, "balance": 0.5}   # 우선순위·배정 기본 가중치(미션 9에서 재튜닝)


def travel_h(team, r, actual=False):
    """도로 굴곡 1.3배, 포장 35km/h·비포장 25km/h, 악천후 20% 감속. actual=True면 실제 도로 지연을 반영."""
    # [TODO : 학생 실습 9] AAR 기반 출동 우선순위 배정 오류 분석 및 가중치 매트릭스 재튜닝 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def on_site_h(team, r):
    """현장 정비시간: 4인 기준 표준시간을 투입 인원으로 환산. 현장 수리 불가면 진단·후송 조치 1시간."""
    # [TODO : 학생 실습 9] AAR 기반 출동 우선순위 배정 오류 분석 및 가중치 매트릭스 재튜닝 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def priority(r, now, est_h, w):
    return w["crit"] * r["crit"] + w["wait"] * (now - r["t_req"]) + w["short"] / (1 + est_h)


def simulate(req, teams, policy="baseline", w=None, cycle_h=1.0, est=None):
    """배정 주기(cycle_h)마다 대기 요청을 정렬해 이동정비반에 배정하고 출동·정비·복귀를 모의한다.
    baseline: 접수 순서 + 거리 최근접 정비반 / priority: 우선순위 점수 순서 + 도착 예상시각·당일 부하 균형"""
    w = w or W0
    est = np.asarray(est if est is not None else req["std_repair_h"].values, float)
    n_days = int(req["day"].max()) + 1
    free = {t["team"]: 0.0 for t in teams}
    load = {t["team"]: np.zeros(n_days + 5) for t in teams}
    recs, out, pending, i, now = req.to_dict("records"), [], [], 0, cycle_h
    while i < len(recs) or pending:
        while i < len(recs) and recs[i]["t_req"] <= now:
            pending.append((i, recs[i])); i += 1
        if policy == "priority":
            pending.sort(key=lambda p: -priority(p[1], now, est[p[0]], w))
        for k, r in pending:
            cand = [t for t in teams if r["equip"] in t["skills"]]
            day = min(int(now // 24), n_days + 4)
            if policy == "baseline":
                tm = min(cand, key=lambda t: np.hypot(t["x"] - r["x"], t["y"] - r["y"]))
            else:
                tm = min(cand, key=lambda t: max(free[t["team"]], now) + travel_h(t, r)
                         + w["balance"] * load[t["team"]][day] / DUTY_H * 4)
            backlog = max(0.0, free[tm["team"]] - now)
            go = max(free[tm["team"]], now)
            tr = travel_h(tm, r, actual=True)
            work = on_site_h(tm, r)
            free[tm["team"]] = go + 2 * tr + work
            load[tm["team"]][min(int(go // 24), n_days + 4)] += 2 * tr + work
            out.append({**r, "idx": k, "team": tm["team"], "dist_km": float(np.hypot(tm["x"] - r["x"], tm["y"] - r["y"])),
                        "backlog_h": backlog, "cycle_wait_h": now - r["t_req"], "dispatch": go, "travel_h": tr,
                        "work_h": work, "response_h": go + tr - r["t_req"], "downtime_h": go + tr + work - r["t_req"]})
        pending, now = [], now + cycle_h
    L = pd.DataFrame({k: v[:n_days] for k, v in load.items()}) / DUTY_H      # 정비반별 일일 부하율
    return pd.DataFrame(out).sort_values("idx").reset_index(drop=True), L


def metrics(res, L):
    c3 = res[res["crit"] == 3]
    return {"평균 도착(h)": round(res["response_h"].mean(), 2), "핵심장비 도착(h)": round(c3["response_h"].mean(), 2),
            "핵심장비 90%(h)": round(c3["response_h"].quantile(0.9), 2), "평균 부하율": round(float(L.values.mean()), 3),
            "부하율 편차": round(float(L.mean().std()), 3), "과부하 일수": int((L > 1.0).values.sum()),
            "총 이동(h)": round(float(res["travel_h"].sum() * 2), 1)}

# ---- 접수 시점 파생 피처(미션 4와 동일) ----
def make_features(req, teams):
    """접수 시점에 알 수 있는 정보만으로 파생 피처를 만든다(미래 정보 누설 방지)."""
    X = pd.DataFrame(index=req.index)
    X["hour"] = req["t_req"] % 24
    X["is_night"] = (X["hour"] >= 19).astype(int)
    X["dow"] = req["day"] % 7
    X["weather_bad"] = (req["weather"] != "clear").astype(int)
    X["unpaved"] = (req["road"] == "unpaved").astype(int)
    X["no_parts"] = (~req["parts_ready"]).astype(int)
    X["std_table_h"] = req["fault"].map(FAULT_H) * req["equip"].map(EQUIP_F)        # 정비 표준시간표 값
    X["crit_x_noparts"] = req["crit"] * X["no_parts"]
    X["bad_x_table"] = X["weather_bad"] * X["std_table_h"]
    X["near_team_km"] = [min(np.hypot(t["x"] - x, t["y"] - y) for t in teams if e in t["skills"]) for x, y, e in zip(req["x"], req["y"], req["equip"])]
    X["hub_km"] = [min(np.hypot(hx - x, hy - y) for hx, hy in HUBS) for x, y in zip(req["x"], req["y"])]
    X["req_24h"] = [((req["t_req"] > t - 24) & (req["t_req"] <= t)).sum() for t in req["t_req"]]
    X["fault_past_mean"] = req.groupby("fault")["std_repair_h"].transform(lambda s: s.shift().expanding().mean()).fillna(req["std_repair_h"].mean())
    X["crit"] = req["crit"]
    for e in EQUIP_F:
        X[f"eq_{e}"] = (req["equip"] == e).astype(int)
    return X


def fit_repair_model(hist, teams):
    """과거 요청으로 정비 소요시간(4인 기준) 예측 모델을 학습한다."""
    return GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=0).fit(make_features(hist, teams), hist["std_repair_h"])

CW = {1: 1.0, 2: 3.0, 3: 6.0}                                    # 중요도별 가동중단 비용 지수
PRICE = {"downtime": 10, "move": 5, "overtime": 2}               # [교육용가상자료] 가상 단가(만원): 비용지수 1, 차량 운행 1h, 초과근무 1인·h
TARGET_H = 4.0


def evaluate(res, L, teams):
    crew = pd.Series({t["team"]: t["crew"] for t in teams})
    over = ((L - 1.0).clip(lower=0) * DUTY_H * crew).values.sum()           # 초과근무 인·시간
    return {"가동중단 비용지수": float((res["crit"].map(CW) * res["downtime_h"]).sum()),
            "핵심장비 도착(h)": float(res.loc[res["crit"] == 3, "response_h"].mean()),
            "초과근무(인·h)": float(over), "차량 운행(h)": float(res["travel_h"].sum() * 2),
            "핵심 4h 초과(건)": int(((res["crit"] == 3) & (res["response_h"] > TARGET_H)).sum())}


def won(m):
    return m["가동중단 비용지수"] * PRICE["downtime"] + m["차량 운행(h)"] * PRICE["move"] + m["초과근무(인·h)"] * PRICE["overtime"]


def objective(m):
    """튜닝 목표: 가상 총비용 + 핵심장비 4시간 초과 1건당 임무 영향 벌점(가상 500만원)."""
    return won(m) + 500 * m["핵심 4h 초과(건)"]


teams = make_teams()
hist, req = make_requests(seed=11), make_requests(seed=1)
model = fit_repair_model(hist, teams)
est_h, est_r = model.predict(make_features(hist, teams)), model.predict(make_features(req, teams))

# 1) AAR: 현행 가중치(W0)로 운용한 평가 기간의 핵심장비 지연 원인 분류
res, L = simulate(req, teams, "priority", w=W0, est=est_r)
late = res[(res["crit"] == 3) & (res["response_h"] > TARGET_H)].copy()
est_tr = [travel_h(next(t for t in teams if t["team"] == tm), r) for tm, r in zip(late["team"], late.to_dict("records"))]
late["원인"] = np.select([late["backlog_h"] > 1.0, late["dist_km"] > 25, late["travel_h"] > np.array(est_tr) * 1.2],
                        ["정비반 부하 집중(앞 작업 대기)", "원거리 정비반 배정", "도로·기상 지연"], "배정 주기 대기")
print(f"[교육용가상자료] AAR: 핵심장비 요청 {int((res['crit'] == 3).sum())}건 중 {TARGET_H:.0f}시간 초과 {len(late)}건")
print(late["원인"].value_counts().to_string())
print("훈련 주간(요청 급증) 발생 비율 {:.0%}, 지연 건의 평균 앞 작업 대기 {:.1f}h".format(late["day"].isin([8, 9, 10, 11, 22, 23, 24]).mean(), late["backlog_h"].mean()))

# 2) 가중치 매트릭스 재튜닝: 과거 기간(hist)에서 탐색 -> 평가 기간(req)에서 검증
grid = []
for c, wt, sh, bl in itertools.product([0.5, 1.0, 2.0, 3.0], [0.1, 0.3, 0.6], [0.2, 1.0], [0.0, 0.5, 1.0, 2.0, 3.0]):
    w = {"crit": c, "wait": wt, "short": sh, "balance": bl}
    r_, L_ = simulate(hist, teams, "priority", w=w, est=est_h)
    grid.append({**w, "cost": objective(evaluate(r_, L_, teams))})
G = pd.DataFrame(grid).sort_values("cost")
W1 = {k: float(G.iloc[0][k]) for k in W0}
print(f"\n가중치 조합 {len(G)}개 탐색(과거 30일). 현행 {W0} -> 재튜닝 {W1}")

rows = {}
rb, Lb = simulate(req, teams, "baseline")
rows["Baseline(거리)"] = evaluate(rb, Lb, teams)
rows["현행 W0"] = evaluate(res, L, teams)
r1, L1 = simulate(req, teams, "priority", w=W1, est=est_r)
rows["재튜닝 W1"] = evaluate(r1, L1, teams)
T = pd.DataFrame(rows).T
T["가상 총비용(만원)"] = [won(m) for m in rows.values()]
print("\n[평가 기간 30일 비교]")
print(T.round(1).to_string())
print(f"핵심장비 {TARGET_H:.0f}시간 초과: 현행 {len(late)}건 -> 재튜닝 {rows['재튜닝 W1']['핵심 4h 초과(건)']}건")
w0, w1 = rows["현행 W0"], rows["재튜닝 W1"]
print(f"재튜닝 효과(현행 대비): 가상 총비용 {won(w1) / won(w0) - 1:+.1%}, 차량 운행 {w1['차량 운행(h)'] / w0['차량 운행(h)'] - 1:+.1%}")
print("AAR 결론: 남은 지연은 전 정비반이 동시에 바쁜 급증 구간의 대기에서 생겨 가중치로는 줄지 않는다"
      " -> 훈련 주간 예고 시 정비반 사전 증원·상급 정비부대 지원 요청을 운영 조치로 권고한다.")

# 3) 3대 정량 효과 지표(Baseline 대비 재튜닝 W1)
b, n = rows["Baseline(거리)"], rows["재튜닝 W1"]
save = (b["가동중단 비용지수"] - n["가동중단 비용지수"]) * PRICE["downtime"]
cut_h = 1 - n["핵심장비 도착(h)"] / b["핵심장비 도착(h)"]
over = b["초과근무(인·h)"] - n["초과근무(인·h)"]
print("\n[3대 정량 효과 지표(30일, Baseline 대비)]")
print(f" 1) 가동중단 비용 절감액: {save:,.0f}만원(가상 단가)")
print(f" 2) 핵심장비 도착시간 단축률: {cut_h:.1%} ({b['핵심장비 도착(h)']:.2f}h -> {n['핵심장비 도착(h)']:.2f}h)")
print(f" 3) 정비인력 초과근무 변화: {over:+,.0f}인·h (양수=감소)")
move = (n["차량 운행(h)"] - b["차량 운행(h)"]) * PRICE["move"]
print(f" 부대비용: 차량 운행 증가 {move:,.0f}만원 -> 순효과 {save + over * PRICE['overtime'] - move:,.0f}만원/월")
print("ROI 로드맵: 단기(1~3개월) 관제 화면·우선순위 시범 -> 중기(6개월) 분기별 AAR 가중치 재튜닝 -> 장기(1년) 정비체계 연동")
print("모든 금액은 교육용 가정 단가로 계산한 상대 비교값이며 실제 예산 근거로 쓰지 않는다.")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
late["원인"].value_counts().rename(index={"정비반 부하 집중(앞 작업 대기)": "team backlog", "원거리 정비반 배정": "far team",
                                         "도로·기상 지연": "road/weather", "배정 주기 대기": "cycle wait"}).plot.bar(ax=ax[0], rot=0)
ax[0].set_title("AAR: causes of late critical response"); ax[0].set_xlabel("")
ax[1].bar(["baseline", "W0", "W1"], T["가상 총비용(만원)"]); ax[1].set_title("Virtual total cost (30 days)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 09 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] AAR: 핵심장비 요청 99건 중 4시간 초과 8건
원인
정비반 부하 집중(앞 작업 대기)    8
훈련 주간(요청 급증) 발생 비율 62%, 지연 건의 평균 앞 작업 대기 3.5h

가중치 조합 120개 탐색(과거 30일). 현행 {'crit': 1.0, 'wait': 0.3, 'short': 0.2, 'balance': 0.5} -> 재튜닝 {'crit': 0.5, 'wait': 0.1, 'short': 0.2, 'balance': 0.0}

[평가 기간 30일 비교]
              가동중단 비용지수  핵심장비 도착(h)  초과근무(인·h)  차량 운행(h)  핵심 4h 초과(건)  가상 총비용(만원)
Baseline(거리)     6494.6         3.1     1018.3     504.7         27.0     69506.0
현행 W0            5055.3         1.9      869.7     744.9          8.0     56017.0
재튜닝 W1           4962.9         1.9      872.4     695.1          8.0     54848.7
핵심장비 4시간 초과: 현행 8건 -> 재튜닝 8건
재튜닝 효과(현행 대비): 가상 총비용 -2.1%, 차량 운행 -6.7%
AAR 결론: 남은 지연은 전 정비반이 동시에 바쁜 급증 구간의 대기에서 생겨 가중치로는 줄지 않는다 -> 훈련 주간 예고 시 정비반 사전 증원·상급 정비부대 지원 요청을 운영 조치로 권고한다.

[3대 정량 효과 지표(30일, Baseline 대비)]
 1) 가동중단 비용 절감액: 15,317만원(가상 단가)
 2) 핵심장비 도착시간 단축률: 40.2% (3.13h -> 1.87h)
 3) 정비인력 초과근무 변화: +146인·h (양수=감소)
 부대비용: 차량 운행 증가 952만원 -> 순효과 14,657만원/월
ROI 로드맵: 단기(1~3개월) 관제 화면·우선순위 시범 -> 중기(6개월) 분기별 AAR 가중치 재튜닝 -> 장기(1년) 정비체계 연동
모든 금액은 교육용 가정 단가로 계산한 상대 비교값이며 실제 예산 근거로 쓰지 않는다.
```
</details>
